In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

C:\Users\Girish Kumar\AppData\Local\Temp\ipykernel_15736\3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [5]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


In [8]:
data.head(5)

,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0


### Let's divide the training and test set into two partitions

In [7]:
data_train= data
data_val= pd.read_csv("../data/kg_test.csv")

In [9]:
X, y= data["text"], data["label"]

In [21]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Data Preprocessing

In [15]:
import string
from nltk.corpus import stopwords
import nltk
#nltk.download("stopwords")

from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [11]:
import re

# <>! AI GENERATED PIPELINE <>!
def clean_html(text):
  text= re.sub(r"<script.*?>.*?</script>", "", text, flags=re.S) # remove html
  text= re.sub(r"<style.*?>.*?</style>", "", text, flags=re.S)   # remove jss
  text= re.sub(r"<!--.*?-->", "", text, flags=re.S)              # remove css
  text= re.sub(r"<[^>]+>", "", text)                             # remove comments
  text= re.sub(r"[^a-zA-Z\s]", " ", text)                        # remove especial ch
  text= re.sub(r"\d+", "", text)                                 # remove numbers
  text= re.sub(r"\b[a-zA-Z]\b", "", text)                        # remove single ch
  text= re.sub(r"^\s+[a-zA-Z]\s+", "", text)                     # remove single ch from start
  text= re.sub(r"\s+", " ", text)                                # substitute multiple spaces with single space
  text= re.sub(r"^b\s+", "", text)                               # remove prefixed 'b'
  text= text.lower()                                             # convert to lowercase

  return text.strip()

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [12]:
data["cleaned_text"]= data["text"].apply(clean_html)
data.head(5)

,text,label,cleaned_text
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1,dear sir strictly private business proposal am...
1,Will do.,0,will do
2,Nora--Cheryl has emailed dozens of memos about...,0,nora cheryl has emailed dozens of memos about ...
3,Dear Sir=2FMadam=2C I know that this proposal ...,1,dear sir fmadam know that this proposal might ...
4,fyi,0,fyi


## Now let's work on removing stopwords
Remove the stopwords.

In [13]:
from nltk.tokenize import word_tokenize

stop_words= set(stopwords.words("english"))
no_stopwords_list= []

for text in data["cleaned_text"]:
  tokens = word_tokenize(text)
  filtered_tokens = []
  for t in tokens:
    if t.lower() not in stop_words:
      filtered_tokens.append(t)
  no_stopwords_list.append(" ".join(filtered_tokens))

data["no_stopwords"] = no_stopwords_list
data.head()

,text,label,cleaned_text,no_stopwords
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1,dear sir strictly private business proposal am...,dear sir strictly private business proposal mi...
1,Will do.,0,will do,
2,Nora--Cheryl has emailed dozens of memos about...,0,nora cheryl has emailed dozens of memos about ...,nora cheryl emailed dozens memos haiti weekend...
3,Dear Sir=2FMadam=2C I know that this proposal ...,1,dear sir fmadam know that this proposal might ...,dear sir fmadam know proposal might surprise e...
4,fyi,0,fyi,fyi


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [16]:
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt") 

[nltk_data] Downloading package wordnet to C:\Users\Girish
[nltk_data]     Kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to C:\Users\Girish
[nltk_data]     Kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Girish
[nltk_data]     Kumar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [18]:
from nltk.stem import WordNetLemmatizer
lemmatizer= WordNetLemmatizer()

lemmatized_list= []

for text in data["no_stopwords"]:
  tokens= word_tokenize(text)
  lemm= []
  for t in tokens:
    lemm.append(lemmatizer.lemmatize(t))
  lemmatized_list.append(" ".join(lemm))

data["lemmatized"]= lemmatized_list
data.head()

,text,label,cleaned_text,no_stopwords,lemmatized
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1,dear sir strictly private business proposal am...,dear sir strictly private business proposal mi...,dear sir strictly private business proposal mi...
1,Will do.,0,will do,,
2,Nora--Cheryl has emailed dozens of memos about...,0,nora cheryl has emailed dozens of memos about ...,nora cheryl emailed dozens memos haiti weekend...,nora cheryl emailed dozen memo haiti weekend p...
3,Dear Sir=2FMadam=2C I know that this proposal ...,1,dear sir fmadam know that this proposal might ...,dear sir fmadam know proposal might surprise e...,dear sir fmadam know proposal might surprise e...
4,fyi,0,fyi,fyi,fyi


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [19]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer= CountVectorizer()
X_bow= vectorizer.fit_transform(data["lemmatized"])
feature_names= vectorizer.get_feature_names_out()

spam_idx= data[data["label"]==1].index
ham_idx=  data[data["label"]==0].index

spam_bow= X_bow[spam_idx, :]
ham_bow=  X_bow[ham_idx, :]

spam_counts= pd.DataFrame(spam_bow.toarray(), columns=feature_names).sum().sort_values(ascending=False)
ham_counts=  pd.DataFrame(ham_bow.toarray(), columns=feature_names).sum().sort_values(ascending=False)

print("Top 10 words in SPAM:")
print(spam_counts.head(10))
print("\nTop 10 words in HAM:")
print(ham_counts.head(10))

Top 10 words in SPAM:
money          981
account        895
bank           800
fund           781
transaction    549
business       514
country        508
mr             489
nbsp           475
million        460
dtype: int64

Top 10 words in HAM:
state        136
pm           127
would        107
president     99
time          95
call          94
mr            91
obama         84
percent       81
secretary     79
dtype: int64


## Extra features

In [22]:
# Convert the split arrays back into DataFrames so we can add new columns
data_train = pd.DataFrame({
    "preprocessed_text": X_train,
    "label": y_train
})

data_val = pd.DataFrame({
    "preprocessed_text": X_test,
    "label": y_test
})

In [23]:
data_train = pd.DataFrame({
    "preprocessed_text": X_train,
    "label": y_train
})

In [24]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,preprocessed_text,label,money_mark,suspicious_words,text_len
442,Dear=2C Good day hope fine=2Cdear am writting ...,1,0,1,1609
962,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,1,1,1,3123
971,Will do.,0,0,0,8
190,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...,1,1,1,530
551,"Dear Friend, My name is LOI C.ESTRADA,The wife...",1,1,1,2126


## How would work the Bag of Words with Count Vectorizer concept?

In [25]:
from sklearn.feature_extraction.text import CountVectorizer   # Importamos CountVectorizer para construir el Bag of Words

# -------------------------------------------------------------------
# 1. Create the CountVectorizer object
#    - max_features limits vocabulary size (optional)
#    - stop_words='english' removes stopwords automatically (optional)
# -------------------------------------------------------------------

vectorizer = CountVectorizer()  
# ↑ CountVectorizer convierte texto en una matriz numérica donde cada columna
#   representa una palabra distinta y cada fila un documento/email.

# -------------------------------------------------------------------
# 2. Fit the vectorizer to the training text and transform it
#    - fit() aprende el vocabulario
#    - transform() convierte texto → matriz de conteos
# -------------------------------------------------------------------

X_train_bow = vectorizer.fit_transform(data_train["preprocessed_text"])
# ↑ Crea la matriz Bag of Words:
#     • filas    → emails
#     • columnas → palabras únicas
#     • valores  → número de veces que la palabra aparece en ese email

# -------------------------------------------------------------------
# 3. Apply the same vocabulary to transform the validation set
#    IMPORTANT: we only call transform(), never fit_transform() on validation
# -------------------------------------------------------------------

X_val_bow = vectorizer.transform(data_val["preprocessed_text"])
# ↑ Usamos el MISMO vocabulario aprendido en entrenamiento.
#   Esto asegura que ambos conjuntos tienen la misma representación numérica.

# -------------------------------------------------------------------
# 4. Optional: Inspect vocabulary size
# -------------------------------------------------------------------

print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
print("Matrix shape (train):", X_train_bow.shape)
print("Matrix shape (val):", X_val_bow.shape)

Vocabulary size: 23568
Matrix shape (train): (800, 23568)
Matrix shape (val): (200, 23568)


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer   # Importamos el vectorizador TF-IDF

tfidf_vectorizer = TfidfVectorizer()  
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train["preprocessed_text"])
X_val_tfidf = tfidf_vectorizer.transform(data_val["preprocessed_text"])

print("TF-IDF training matrix shape:", X_train_tfidf.shape)
print("TF-IDF validation matrix shape:", X_val_tfidf.shape)

TF-IDF training matrix shape: (800, 23568)
TF-IDF validation matrix shape: (200, 23568)


## And the Train a Classifier?

In [28]:
from sklearn.linear_model import LogisticRegression    # Importamos el modelo
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

clf = LogisticRegression(max_iter=1000)   


clf.fit(X_train_tfidf, data_train["label"])  
y_pred = clf.predict(X_val_tfidf)  
acc = accuracy_score(data_val["label"], y_pred)  

print("Accuracy:", acc)

print("\nClassification Report:")
print(classification_report(data_val["label"], y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(data_val["label"], y_pred))


Accuracy: 0.975

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       112
           1       0.99      0.95      0.97        88

    accuracy                           0.97       200
   macro avg       0.98      0.97      0.97       200
weighted avg       0.98      0.97      0.97       200


Confusion Matrix:
[[111   1]
 [  4  84]]


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [29]:
from sklearn.naive_bayes import MultinomialNB           # Required classifier (default params ONLY)
from sklearn.metrics import accuracy_score              # To measure accuracy
from scipy.sparse import hstack                         # To combine sparse matrices (BoW/TF-IDF + extra features)
from sklearn.preprocessing import MinMaxScaler          # Ensures extra features stay non-negative

import numpy as np

# -------------------------------------------------------------------
# 1. Helper function to train & evaluate MultinomialNB
# -------------------------------------------------------------------

def evaluate_model(X_train, X_val, y_train, y_val):
    """
    Trains MultinomialNB (default params) and prints accuracy.
    Only the feature matrix changes between experiments.
    """
    
    model = MultinomialNB()   # Classifier MUST remain unchanged (default params)
    
    model.fit(X_train, y_train)   # Train on the given features
    
    y_pred = model.predict(X_val) # Predict on validation set
    
    acc = accuracy_score(y_val, y_pred)   # Compute accuracy
    
    print("Accuracy:", acc)               # Display results
    
    return acc


# -------------------------------------------------------------------
# 2. Build EXTRA FEATURES matrices (money_mark, suspicious_words, text_len)
# -------------------------------------------------------------------

extra_train = np.array(data_train[["money_mark","suspicious_words","text_len"]])
extra_val   = np.array(data_val[["money_mark","suspicious_words","text_len"]])

# -------------------------------------------------------------------
# 3. SCALE EXTRA FEATURES (using MinMaxScaler → required for MultinomialNB)
#    This ensures NO negative values are created.
# -------------------------------------------------------------------

scaler = MinMaxScaler()               # Scale features into [0,1] range (safe for Naive Bayes)

extra_train_scaled = scaler.fit_transform(extra_train)   # Fit on training, transform
extra_val_scaled   = scaler.transform(extra_val)         # Transform validation only

# -------------------------------------------------------------------
# 4. COMBINE sparse text matrices (BoW or TF-IDF) with extra features
# -------------------------------------------------------------------

X_train_bow_extra = hstack([X_train_bow, extra_train_scaled])      # BoW + extra
X_val_bow_extra   = hstack([X_val_bow, extra_val_scaled])

X_train_tfidf_extra = hstack([X_train_tfidf, extra_train_scaled])  # TF-IDF + extra
X_val_tfidf_extra   = hstack([X_val_tfidf, extra_val_scaled])


# -------------------------------------------------------------------
# 5. RUN ALL EXPERIMENTS (4 feature configurations)
# -------------------------------------------------------------------

print("\n--- Bag of Words only ---")
evaluate_model(X_train_bow, X_val_bow, data_train["label"], data_val["label"])

print("\n--- TF-IDF only ---")
evaluate_model(X_train_tfidf, X_val_tfidf, data_train["label"], data_val["label"])

print("\n--- Bag of Words + Extra Features ---")
evaluate_model(X_train_bow_extra, X_val_bow_extra, data_train["label"], data_val["label"])

print("\n--- TF-IDF + Extra Features ---")
evaluate_model(X_train_tfidf_extra, X_val_tfidf_extra, data_train["label"], data_val["label"])


--- Bag of Words only ---
Accuracy: 0.92

--- TF-IDF only ---
Accuracy: 0.9

--- Bag of Words + Extra Features ---
Accuracy: 0.92

--- TF-IDF + Extra Features ---
Accuracy: 0.9


0.9

In [30]:
data_train = pd.DataFrame({
    "preprocessed_text": X_train,
    "label": y_train
})

data_val = pd.DataFrame({
    "preprocessed_text": X_test,
    "label": y_test
})

data_train.head()

,preprocessed_text,label
442,Dear=2C Good day hope fine=2Cdear am writting ...,1
962,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,1
971,Will do.,0
190,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...,1
551,"Dear Friend, My name is LOI C.ESTRADA,The wife...",1


In [31]:
money_simbol_list = "|".join(["euro", "dollar", "pound", "€", r"\$"])

suspicious_words = "|".join([
    "free","cheap","sex","money","account","bank",
    "fund","transfer","transaction","win","deposit","password"
])


data_train["money_mark"] = data_train["preprocessed_text"].str.contains(money_simbol_list) * 1


data_train["suspicious_words"] = data_train["preprocessed_text"].str.contains(suspicious_words) * 1


data_train["text_len"] = data_train["preprocessed_text"].apply(len)


data_val["money_mark"] = data_val["preprocessed_text"].str.contains(money_simbol_list) * 1
data_val["suspicious_words"] = data_val["preprocessed_text"].str.contains(suspicious_words) * 1
data_val["text_len"] = data_val["preprocessed_text"].apply(len)

data_train.head() 

,preprocessed_text,label,money_mark,suspicious_words,text_len
442,Dear=2C Good day hope fine=2Cdear am writting ...,1,0,1,1609
962,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,1,1,1,3123
971,Will do.,0,0,0,8
190,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...,1,1,1,530
551,"Dear Friend, My name is LOI C.ESTRADA,The wife...",1,1,1,2126


In [32]:
from sklearn.preprocessing import MinMaxScaler  
import numpy as np


extra_train = np.array(data_train[["money_mark","suspicious_words","text_len"]])
extra_val   = np.array(data_val[["money_mark","suspicious_words","text_len"]])



scaler = MinMaxScaler()
extra_train_scaled = scaler.fit_transform(extra_train)   # Ajusta en train
extra_val_scaled   = scaler.transform(extra_val)         # Transforma val

In [33]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


bow = CountVectorizer()                                          
X_train_bow = bow.fit_transform(data_train["preprocessed_text"]) 
X_val_bow   = bow.transform(data_val["preprocessed_text"])       


tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(data_train["preprocessed_text"])  
X_val_tfidf   = tfidf.transform(data_val["preprocessed_text"])        

In [34]:
from scipy.sparse import hstack   


X_train_bow_extra = hstack([X_train_bow, extra_train_scaled])      
X_val_bow_extra   = hstack([X_val_bow, extra_val_scaled])

X_train_tfidf_extra = hstack([X_train_tfidf, extra_train_scaled])  
X_val_tfidf_extra   = hstack([X_val_tfidf, extra_val_scaled])

In [35]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

def evaluate_model(X_train, X_val, y_train, y_val):
    model = MultinomialNB()     
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)  # Predicciones
    
    acc = accuracy_score(y_val, y_pred)  # Accuracy
    
    print("Accuracy:", acc)
    return acc


print("\n--- Bag of Words SOLO ---")
evaluate_model(X_train_bow, X_val_bow, data_train["label"], data_val["label"])

print("\n--- TF-IDF SOLO ---")
evaluate_model(X_train_tfidf, X_val_tfidf, data_train["label"], data_val["label"])

print("\n--- Bag of Words + Extra Features ---")
evaluate_model(X_train_bow_extra, X_val_bow_extra, data_train["label"], data_val["label"])

print("\n--- TF-IDF + Extra Features ---")
evaluate_model(X_train_tfidf_extra, X_val_tfidf_extra, data_train["label"], data_val["label"])


--- Bag of Words SOLO ---
Accuracy: 0.92

--- TF-IDF SOLO ---
Accuracy: 0.9

--- Bag of Words + Extra Features ---
Accuracy: 0.92

--- TF-IDF + Extra Features ---
Accuracy: 0.9


0.9